In [1]:
!pip install uv

In [2]:
!uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community


Using Python 3.9.25 environment at: C:\Users\fredd\OneDrive\Documents\LLMOPs\.venv
Audited 6 packages in 2.29s


In [3]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


Data Ingestion

In [4]:
from langchain.document_loaders import TextLoader

In [5]:
loader = TextLoader("C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt", encoding="utf8")
documents = loader.load()
documents[0].page_content[:500]

'How AI broke the smart home in 2025 \nThe arrival of generative AI assistants in our smart homes held such promise; instead, they struggle to turn on the lights.\n\nThis morning, I asked my Alexa-enabled Bosch coffee machine to make me a coffee. Instead of running my routine, it told me it couldn’t do that. Ever since I upgraded to Alexa Plus, Amazon’s generative-AI-powered voice assistant, it has failed to reliably run my coffee routine, coming up with a different excuse almost every time I ask.\n\n'

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
text_chunks=text_splitter.split_documents(documents)
text_chunks


[Document(metadata={'source': 'C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt'}, page_content='How AI broke the smart home in 2025 \nThe arrival of generative AI assistants in our smart homes held such promise; instead, they struggle to turn on the lights.'),
 Document(metadata={'source': 'C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt'}, page_content='This morning, I asked my Alexa-enabled Bosch coffee machine to make me a coffee. Instead of running my routine, it told me it couldn’t do that. Ever since I upgraded to Alexa Plus, Amazon’s'),
 Document(metadata={'source': 'C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt'}, page_content='Plus, Amazon’s generative-AI-powered voice assistant, it has failed to reliably run my coffee routine, coming up with a different excuse almost every time I ask.'),
 Document(metadata={'source': 'C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt'}, page_content='It’s 2025, and AI still can’t rel

In [7]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [8]:
embeddings=OpenAIEmbeddings()


C:\Users\fredd\AppData\Local\Temp\ipykernel_23496\2760764271.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings=OpenAIEmbeddings()


In [9]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)


In [10]:
vectorstore


In [11]:
retriever=vectorstore.as_retriever()


In [12]:
# Perform similarity search
query = "What is the Key Characteristics of AI in 2025?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
How AI broke the smart home in 2025 
The arrival of generative AI assistants in our smart homes held such promise; instead, they struggle to turn on the lights.
--------------------------------------------------
Document 2:
It’s 2025, and AI still can’t reliably control my smart home. I’m beginning to wonder if it ever will.
--------------------------------------------------
Document 3:
The potential for generative AI and large language models to take the complexity out of the smart home, making it easier to set up, use, and manage connected devices, is compelling. So is the promise
--------------------------------------------------
Document 4:
AI companies want a new internet — and they think they’ve found the key
AI can’t even turn on the lights
--------------------------------------------------


In [13]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt=ChatPromptTemplate.from_template(template)
prompt


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [14]:
from langchain.schema.output_parser import StrOutputParser
output_parser=StrOutputParser()


In [15]:
from langchain.chat_models import ChatOpenAI
llm_model=ChatOpenAI(model_name="gpt-4o-mini")


C:\Users\fredd\AppData\Local\Temp\ipykernel_23496\1726283732.py:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm_model=ChatOpenAI(model_name="gpt-4o-mini")


In [17]:
from langchain.schema.runnable import RunnablePassthrough

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)


In [18]:
rag_chain.invoke("tell me about AI trends in 2025")

"In 2025, AI trends are heavily focused on the integration of generative AI assistants into smart homes. However, there are significant challenges, as these AI systems struggle to perform basic tasks, such as turning on lights reliably. The initial promise of simplifying the management of connected devices has not been fully realized. AI companies are exploring the need for a new internet infrastructure to support these advancements. Despite the potential for large language models to enhance user experience by reducing complexity, practical applications remain limited. This situation raises questions about the reliability and effectiveness of AI in everyday environments. The trends indicate a gap between potential and reality in AI's role in smart homes. Overall, while advancements are being pursued, users are left wondering about the true capabilities of AI in their homes."